# M127 â€” engine optimization verification

Build measurable evidence for accepting or rejecting M127 (copula flip,
distribution registry, column dispatcher, validator split, code cleanup).

**How to use.** Run once before M127 lands (baseline capture); save the
printed report as `analysis/m127_pre.md`. Run again after M127 lands; save
as `analysis/m127_post.md`. Operator compares â€” this notebook does not
assert pass/fail.

Spec: `project/missions/mission-127v-verification-notebook.md`.


## Cell 0 â€” Setup and configs

Load the five bundled builder templates, build a stress config, build a
degenerate (poisson Î»â†’0) config. All keyed in one dict for the rest of
the notebook to iterate.

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy import stats

from plotsim import (
    create,
    create_from_yaml,
    generate_tables,
    generate_tables_with_state,
    load_config,
    validate,
)

SEED = 42
ALT_SEED = 43

REPO_ROOT = Path.cwd()
# Walk up to project root (where pyproject.toml lives) so the notebook works
# whether launched from the repo root or from analysis/.
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
TEMPLATE_DIR = REPO_ROOT / "plotsim" / "configs" / "new"
TEMPLATE_NAMES = ["saas", "retail", "education", "marketing", "hr"]


def _load_template(name, seed):
    raw = yaml.safe_load((TEMPLATE_DIR / f"{name}_template.yaml").read_text(encoding="utf-8"))
    if isinstance(raw.get("window"), dict):
        for k in ("start", "end"):
            v = raw["window"].get(k)
            if v is not None and not isinstance(v, str):
                raw["window"][k] = str(v)
    raw["seed"] = seed
    return create(**raw)


CONFIGS: dict = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for tname in TEMPLATE_NAMES:
        CONFIGS[tname] = _load_template(tname, SEED)

    CONFIGS["stress"] = create(
        about="Stress: 1000 entities x 20 metrics x 4 correlations x 48 periods",
        unit="account",
        window=("2022-01", "2025-12", "monthly"),
        metrics=[{"name": f"m{i:02d}", "type": "score", "polarity": "positive"}
                 for i in range(20)],
        segments=[
            {"name": "growth_seg",  "count": 500, "archetype": "growth"},
            {"name": "decline_seg", "count": 500, "archetype": "decline"},
        ],
        connections=[
            ("m00", "mirrors",   "m01"),
            ("m02", "opposes",   "m03"),
            ("m04", "driven_by", "m05"),
            ("m06", "resists",   "m07"),
        ],
        seed=SEED,
    )

    CONFIGS["degenerate"] = create(
        about="Poisson lambda-zero edge case",
        unit="account",
        window=("2024-01", "2025-12", "monthly"),
        metrics=[
            {"name": "score_a",    "type": "score", "polarity": "positive"},
            {"name": "rare_event", "type": "count", "polarity": "positive"},
        ],
        segments=[
            {"name": "near_zero", "count": 100, "archetype": "decline"},
        ],
        connections=[("score_a", "mirrors", "rare_event")],
        seed=SEED,
    )

print(f"Loaded {len(CONFIGS)} configs:", list(CONFIGS.keys()))
for n, c in CONFIGS.items():
    print(f"  {n:<12} entities={len(c.entities):>4}  metrics={len(c.metrics):>2}  "
          f"correlations={len(c.correlations):>2}  periods={c.time_window.period_count()}")


Loaded 7 configs: ['saas', 'retail', 'education', 'marketing', 'hr', 'stress', 'degenerate']
  saas         entities=  95  metrics= 6  correlations= 3  periods=24
  retail       entities= 110  metrics= 7  correlations= 5  periods=24
  education    entities= 100  metrics= 6  correlations= 5  periods=24
  marketing    entities=  93  metrics= 8  correlations= 6  periods=24
  hr           entities=  95  metrics= 6  correlations= 4  periods=24
  stress       entities=1000  metrics=20  correlations= 4  periods=48
  degenerate   entities= 100  metrics= 2  correlations= 1  periods=24


Config summary: 95 entities × 24 periods = 2,280 cells, 6 metrics, 9 tables. Estimated peak memory: ~100 MB. Expected event rows (upper bound): ~11,400.
Config summary: 110 entities × 24 periods = 2,640 cells, 7 metrics, 10 tables. Estimated peak memory: ~101 MB. Expected event rows (upper bound): ~15,840.
Config summary: 100 entities × 24 periods = 2,400 cells, 6 metrics, 9 tables. Estimated peak memory: ~100 MB. Expected event rows (upper bound): ~7,200.
Config summary: 93 entities × 24 periods = 2,232 cells, 8 metrics, 10 tables. Estimated peak memory: ~100 MB. Expected event rows (upper bound): ~17,856.
Config summary: 95 entities × 24 periods = 2,280 cells, 6 metrics, 9 tables. Estimated peak memory: ~100 MB. Expected event rows (upper bound): ~9,120.
Config summary: 1,000 entities × 48 periods = 48,000 cells, 20 metrics, 3 tables. Estimated peak memory: ~116 MB.
Config summary: 100 entities × 24 periods = 2,400 cells, 2 metrics, 3 tables. Estimated peak memory: ~100 MB.


## Helpers

Small utilities used by multiple cells. Defined once so each measurement
cell can stay focused on its question.

In [2]:
def _metric_cols(df, cfg):
    metric_names = {m.name for m in cfg.metrics}
    return [c for c in df.columns if c in metric_names]


def _primary_fct(tables):
    fct_tables = sorted(t for t in tables if t.startswith("fct_"))
    return fct_tables[0] if fct_tables else None


def _primary_dim(tables, cfg=None):
    # Prefer dim_<entity_type> when the config tells us â€” that is the
    # entity-grain dim with one row per cfg.entities entry. Otherwise
    # fall back to the first non-date dim (engine-direct configs).
    if cfg is not None:
        target = f"dim_{cfg.domain.entity_type}"
        if target in tables:
            return target
    return next(
        (t for t in tables if t.startswith("dim_") and t != "dim_date"),
        None,
    )


def _dim_pk_column(dim_df):
    cands = [c for c in dim_df.columns if c.endswith("_id") and dim_df[c].is_unique]
    return cands[0] if cands else None


def _switch_mode(cfg, mode):
    return cfg.model_copy(update={"generation_mode": mode})


def _switch_seed(cfg, seed):
    return cfg.model_copy(update={"seed": seed})


## Cell 1 â€” Statistical profile per template

For each fact metric column: mean, std, min, max, null count, dtype.
Stored per template â€” pre vs post M127 diff > 10% on any metric mean
flags drift.

In [3]:
def _profile(cfg, tables):
    rows = []
    for tname, df in tables.items():
        if not tname.startswith("fct_"):
            continue
        for col in _metric_cols(df, cfg):
            s = df[col].astype(float)
            rows.append({
                "table":  tname,
                "metric": col,
                "mean":   float(s.mean()),
                "std":    float(s.std(ddof=0)),
                "min":    float(s.min()),
                "max":    float(s.max()),
                "nulls":  int(df[col].isna().sum()),
                "dtype":  str(df[col].dtype),
            })
    return pd.DataFrame(rows)


PROFILES: dict = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, cfg in CONFIGS.items():
        tables = generate_tables(cfg, np.random.default_rng(SEED))
        PROFILES[name] = _profile(cfg, tables)
        print(f"\n--- {name} ---")
        print(PROFILES[name].to_string(index=False))



--- saas ---
              table           metric         mean          std         min     max  nulls   dtype
     fct_engagement feature_adoption     0.395170     0.332888    0.000000     1.0      0 float64
        fct_revenue              mrr 23631.985690 12904.395177  460.260729 50000.0      0 float64
fct_support_tickets       churn_risk     0.597251     0.329789    0.000000     1.0      0 float64
fct_support_tickets              nps    -0.530968    32.741202 -100.000000   100.0      0 float64

--- retail ---
           table               metric       mean        std    min    max  nulls   dtype
    fct_sessions      conversion_rate   0.426008   0.272440    0.0    1.0      0 float64
   fct_purchases           cart_value 827.257130 549.116551   10.0 2000.0      0 float64
   fct_purchases          return_rate   0.535596   0.343928    0.0    1.0      0 float64
   fct_purchases        loyalty_score   0.474548   0.343776    0.0    1.0      0 float64
   fct_purchases repeat_purchase_ra

## Cell 2 â€” Shape recovery per archetype

For each archetype group: pull master trajectory from `state.trajectories`,
join the fact table to the entity dim by FK, group by date_key â†’ period
mean per metric, Pearson r against the trajectory.

Pearson is only meaningful when both series have variance, so flat
archetypes show NaN â€” that is expected, not a defect.

In [4]:
SHAPE_ROWS = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, cfg in CONFIGS.items():
        tables, state = generate_tables_with_state(
            cfg, np.random.default_rng(SEED),
        )
        fct_name = _primary_fct(tables)
        if fct_name is None:
            continue
        fct = tables[fct_name]
        dim_name = _primary_dim(tables, cfg)
        if dim_name is None:
            continue
        dim = tables[dim_name]
        m_cols = _metric_cols(fct, cfg)
        # Use the natural entity key (dim_<unit>_id), not the SCD surrogate
        # (dim_row_id). This is the column the fct actually FKs against.
        fk_col = f"{cfg.domain.entity_type}_id"
        if fk_col not in fct.columns or fk_col not in dim.columns or not m_cols:
            continue
        # SCD2 dims have multiple rows per entity â€” dedupe to one row per
        # natural ID. The engine emits entities in cfg.entities order, so
        # the first appearance of each ID is row i for cfg.entities[i].
        dedup = dim.drop_duplicates(subset=fk_col, keep="first").reset_index(drop=True)
        if len(dedup) != len(cfg.entities):
            continue
        id_to_arch = {dedup[fk_col].iloc[i]: cfg.entities[i].archetype
                      for i in range(len(dedup))}
        id_to_name = {dedup[fk_col].iloc[i]: cfg.entities[i].name
                      for i in range(len(dedup))}
        fct_arch = fct.assign(_archetype=fct[fk_col].map(id_to_arch))

        arch_seen: set = set()
        for ent in cfg.entities:
            if ent.archetype in arch_seen:
                continue
            arch_seen.add(ent.archetype)
            traj = state.trajectories.get(ent.name)
            if traj is None:
                continue
            traj = np.asarray(traj, dtype=float)
            sub = fct_arch[fct_arch["_archetype"] == ent.archetype]
            if sub.empty:
                continue
            means = sub.groupby("date_key")[m_cols].mean().sort_index()
            if len(means) != len(traj):
                continue
            for m in m_cols:
                vals = means[m].astype(float).to_numpy()
                if np.std(vals) == 0 or np.std(traj) == 0:
                    r = float("nan")
                else:
                    r = float(stats.pearsonr(traj, vals).statistic)
                arch_label = ent.archetype
                if len(arch_label) > 55:
                    arch_label = arch_label[:52] + "..."
                SHAPE_ROWS.append({
                    "template":  name,
                    "archetype": arch_label,
                    "metric":    m,
                    "pearson_r": r,
                })

shape_df = pd.DataFrame(SHAPE_ROWS)
print(shape_df.to_string(index=False))


  template            archetype               metric  pearson_r
      saas     promising_client     feature_adoption   0.997392
      saas    steady_enterprise     feature_adoption   0.995949
      saas           slow_churn     feature_adoption   0.988288
      saas    seasonal_accounts     feature_adoption   0.991124
      saas              dormant     feature_adoption        NaN
      saas           turnaround     feature_adoption   0.990249
    retail       loyal_climbers           cart_value   0.895967
    retail       loyal_climbers          return_rate  -0.997206
    retail       loyal_climbers        loyalty_score   0.996532
    retail       loyal_climbers repeat_purchase_rate   0.991557
    retail     holiday_shoppers           cart_value   0.933552
    retail     holiday_shoppers          return_rate  -0.993706
    retail     holiday_shoppers        loyalty_score   0.994850
    retail     holiday_shoppers repeat_purchase_rate   0.827028
    retail           cooled_off         

## Cell 3 â€” Correlation sign matches

For every configured correlation pair: realized Pearson on the fact table.
Match if sign agrees with the configured coefficient (zero-coefficient
pairs would have been dropped at builder time, so all sign expectations
are non-zero).

In [5]:
SIGN_ROWS = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, cfg in CONFIGS.items():
        if not cfg.correlations:
            continue
        tables = generate_tables(cfg, np.random.default_rng(SEED))
        fct_name = _primary_fct(tables)
        if fct_name is None:
            continue
        fct = tables[fct_name]
        for pair in cfg.correlations:
            ma, mb = pair.metric_a, pair.metric_b
            if ma not in fct.columns or mb not in fct.columns:
                continue
            a = fct[ma].astype(float).to_numpy()
            b = fct[mb].astype(float).to_numpy()
            if np.std(a) == 0 or np.std(b) == 0:
                r = float("nan")
            else:
                r = float(stats.pearsonr(a, b).statistic)
            cfg_sign = "+" if pair.coefficient > 0 else "-"
            if np.isnan(r):
                real_sign = "?"
            else:
                real_sign = "+" if r > 0 else ("-" if r < 0 else "0")
            SIGN_ROWS.append({
                "template":        name,
                "metric_a":        ma,
                "metric_b":        mb,
                "configured_coef": pair.coefficient,
                "realized_r":      round(r, 3) if not np.isnan(r) else r,
                "configured_sign": cfg_sign,
                "realized_sign":   real_sign,
                "match":           cfg_sign == real_sign,
            })

sign_df = pd.DataFrame(SIGN_ROWS)
print(sign_df.to_string(index=False))
print()
if not sign_df.empty:
    summary = sign_df.groupby("template")["match"].agg(["sum", "count"])
    summary.columns = ["matches", "total"]
    summary["match_rate"] = summary["matches"] / summary["total"]
    print("Per-template sign-match summary:")
    print(summary.to_string())


  template    metric_a        metric_b  configured_coef  realized_r configured_sign realized_sign  match
    retail  cart_value   loyalty_score             0.40       0.727               +             +   True
    retail return_rate   loyalty_score            -0.55      -0.774               -             -   True
 marketing bounce_rate conversion_rate            -0.55      -0.392               -             -   True
    stress         m00             m01             0.75       0.754               +             +   True
    stress         m02             m03            -0.55       0.617               -             +  False
    stress         m04             m05             0.55       0.717               +             +   True
    stress         m06             m07            -0.40       0.621               -             +  False
degenerate     score_a      rare_event             0.75       0.466               +             +   True

Per-template sign-match summary:
            matches  

## Cell 4 â€” Degenerate-center behavior

The degenerate config places a positive-polarity poisson on a `decline`
archetype â€” trajectory drops to ~0, scaled Î» approaches 0, which fires
the pre-M127 bypass. Both eras must produce finite output, but the
realized correlation between the two metrics should differ:

- **Pre-M127:** bypass fires â†’ independent draw â†’ realized r â‰ˆ 0.
- **Post-M127:** copula applies even at Î»â†’0 â†’ realized r > 0 (configured `mirrors` = 0.75).

Determinism check: same (config, seed) twice â†’ byte-identical tables.

In [6]:
deg_cfg = CONFIGS["degenerate"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tables_a = generate_tables(deg_cfg, np.random.default_rng(SEED))
    tables_b = generate_tables(deg_cfg, np.random.default_rng(SEED))

deterministic_match = all(tables_a[t].equals(tables_b[t]) for t in tables_a)

fct_name = _primary_fct(tables_a)
fct = tables_a[fct_name]
rare = fct["rare_event"].astype(float)
score = fct["score_a"].astype(float)

if rare.std() > 0 and score.std() > 0:
    r = float(stats.pearsonr(score.to_numpy(), rare.to_numpy()).statistic)
else:
    r = float("nan")

DEGENERATE = pd.DataFrame([{
    "rare_event_nan_count":     int(rare.isna().sum()),
    "rare_event_inf_count":     int(np.isinf(rare).sum()),
    "rare_event_mean":          float(rare.mean()),
    "rare_event_std":           float(rare.std(ddof=0)),
    "rare_event_min":           float(rare.min()),
    "rare_event_max":           float(rare.max()),
    "score_x_rare_event_pearson": round(r, 4) if not np.isnan(r) else r,
    "deterministic_repeat":     deterministic_match,
}])
print(DEGENERATE.T.to_string(header=False))


rare_event_nan_count               0
rare_event_inf_count               0
rare_event_mean             2.159583
rare_event_std              1.925474
rare_event_min                   0.0
rare_event_max                  11.0
score_x_rare_event_pearson    0.4658
deterministic_repeat            True


## Cell 5 â€” Determinism per template

Same seed twice â†’ byte-identical tables (every dim/fact/event).
Different seed â†’ fact values differ but shapes match.

These are reported, not asserted â€” the operator decides whether a
mismatch is an acceptable RNG-order shift or a regression.

In [7]:
DET_ROWS = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, cfg in CONFIGS.items():
        a = generate_tables(cfg, np.random.default_rng(SEED))
        b = generate_tables(cfg, np.random.default_rng(SEED))
        same_seed_identical = all(a[t].equals(b[t]) for t in a)

        cfg_alt = _switch_seed(cfg, ALT_SEED)
        c = generate_tables(cfg_alt, np.random.default_rng(ALT_SEED))

        fct_tables = [t for t in a if t.startswith("fct_")]
        any_diff = False
        all_same_shape = True
        for t in fct_tables:
            if a[t].shape != c[t].shape:
                all_same_shape = False
            if not a[t].equals(c[t]):
                any_diff = True

        DET_ROWS.append({
            "template":             name,
            "same_seed_identical":  same_seed_identical,
            "alt_seed_differs":     any_diff,
            "alt_seed_same_shape":  all_same_shape,
        })

det_df = pd.DataFrame(DET_ROWS)
print(det_df.to_string(index=False))


  template  same_seed_identical  alt_seed_differs  alt_seed_same_shape
      saas                 True              True                 True
    retail                 True              True                 True
 education                 True              True                 True
 marketing                 True              True                 True
        hr                 True              True                 True
    stress                 True              True                 True
degenerate                 True              True                 True


## Cell 6 â€” FakerSource RNG parity (serial vs vectorized)

Faker columns must be byte-identical across modes â€” the column
dispatcher routes them through a scalar path regardless. Any difference
means M127 broke FakerSource routing.

Compares string/object columns on every per-entity dim across modes.

In [8]:
def _dim_string_columns(tables):
    out: dict = {}
    for tname, df in tables.items():
        if not tname.startswith("dim_") or tname == "dim_date":
            continue
        cols: dict = {}
        for col in df.columns:
            if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
                cols[col] = tuple(df[col].astype(str).tolist())
        out[tname] = cols
    return out


FAKER_ROWS = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for name, cfg in CONFIGS.items():
        cfg_s = _switch_mode(cfg, "serial")
        s_tables = generate_tables(cfg_s, np.random.default_rng(SEED))
        s_dims = _dim_string_columns(s_tables)

        try:
            cfg_v = _switch_mode(cfg, "vectorized")
            v_tables = generate_tables(cfg_v, np.random.default_rng(SEED))
            v_dims = _dim_string_columns(v_tables)
        except Exception as e:
            FAKER_ROWS.append({
                "template": name, "table": "(error)", "column": "(error)",
                "match": False,
                "note": f"vectorized mode failed: {type(e).__name__}",
            })
            continue

        for tname, scols in s_dims.items():
            vcols = v_dims.get(tname, {})
            for col, sval in scols.items():
                vval = vcols.get(col)
                FAKER_ROWS.append({
                    "template": name, "table": tname, "column": col,
                    "match": sval == vval,
                    "note": "" if sval == vval else f"serial[0]={sval[0]!r}, vec[0]={(vval[0] if vval else None)!r}",
                })

faker_df = pd.DataFrame(FAKER_ROWS)
print(faker_df.to_string(index=False))
print()
print(f"Mismatches: {int((~faker_df['match']).sum())} / {len(faker_df)}")


  template                table            column  match note
      saas             dim_plan           plan_id   True     
      saas             dim_plan         plan_name   True     
      saas          dim_company        company_id   True     
      saas          dim_company      company_name   True     
      saas          dim_company          industry   True     
      saas          dim_company         plan_tier   True     
      saas             dim_user           user_id   True     
      saas             dim_user        company_id   True     
      saas             dim_user         user_name   True     
      saas             dim_user              role   True     
    retail dim_product_category       category_id   True     
    retail dim_product_category     category_name   True     
    retail dim_product_category       margin_tier   True     
    retail          dim_channel        channel_id   True     
    retail          dim_channel      channel_name   True     
    reta

## Cell 7 â€” Validation parity

Intentionally bad configs feed `create()` â€” the validator split in M127
should produce the same exception classes and similar messages. Compare
the pre and post tables row-for-row.

In [9]:
BAD_CONFIGS = [
    ("missing_polarity", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "score"}],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth"}],
    }),
    ("unknown_archetype_shape", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "score", "polarity": "positive"}],
        "segments": [{"name": "s1", "count": 5, "archetype": "exploding_donkey"}],
    }),
    ("duplicate_metric_name", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [
            {"name": "m1", "type": "score", "polarity": "positive"},
            {"name": "m1", "type": "count", "polarity": "positive"},
        ],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth"}],
    }),
    ("connection_to_nonexistent_metric", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "score", "polarity": "positive"}],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth"}],
        "connections": [("m1", "mirrors", "ghost")],
    }),
    ("reserved_plus_in_archetype_dsl", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "score", "polarity": "positive"}],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth + decline"}],
    }),
    ("count_with_range", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "count", "polarity": "positive", "range": [0, 10]}],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth"}],
    }),
    ("amount_without_range", {
        "about": "test", "unit": "x", "window": ("2024-01", "2024-12"),
        "metrics": [{"name": "m1", "type": "amount", "polarity": "positive"}],
        "segments": [{"name": "s1", "count": 5, "archetype": "growth"}],
    }),
]

VAL_ROWS = []
for cname, kwargs in BAD_CONFIGS:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            create(**kwargs)
        VAL_ROWS.append({
            "config_name":   cname,
            "error_type":    "(none)",
            "error_message": "(no exception raised)",
        })
    except Exception as e:
        VAL_ROWS.append({
            "config_name":   cname,
            "error_type":    type(e).__name__,
            "error_message": str(e)[:100].replace("\n", " "),
        })

val_df = pd.DataFrame(VAL_ROWS)
print(val_df.to_string(index=False))


                     config_name      error_type                                                                                        error_message
                missing_polarity ValidationError 1 validation error for UserInput metrics.0.polarity   Field required [type=missing, input_value={'na
         unknown_archetype_shape ValidationError 1 validation error for UserInput   Value error, segment 's1' archetype 'exploding_donkey': archetype
           duplicate_metric_name ValidationError 1 validation error for UserInput   Value error, duplicate metric name(s): ['m1'] [type=value_error, 
connection_to_nonexistent_metric ValidationError 1 validation error for UserInput   Value error, connection 'm1' mirrors 'ghost': endpoint 'ghost' is
  reserved_plus_in_archetype_dsl ValidationError 1 validation error for UserInput   Value error, segment 's1' archetype 'growth + decline': Layered p
                count_with_range ValidationError 1 validation error for UserInput metrics.0   Value 

## Cell 8 â€” Performance timing

Median of three runs per cell. Copula cost % isolates the correlation
contribution by re-running stress with `connections=[]`.

Pre-M127 reading: copula is 92-94% of stress cost. Post-M127 should be
significantly lower; the speedup ratio should hold or improve.

In [10]:
def _time_median(cfg, n=3):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            generate_tables(cfg, np.random.default_rng(SEED))
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


PERF_ROWS = []
for name in ("saas", "stress"):
    cfg = CONFIGS[name]
    cfg_s = _switch_mode(cfg, "serial")
    cfg_v = _switch_mode(cfg, "vectorized")
    t_serial = _time_median(cfg_s)
    try:
        t_vec = _time_median(cfg_v)
    except Exception:
        t_vec = float("nan")
    speedup = (t_serial / t_vec) if t_vec and not np.isnan(t_vec) else float("nan")

    copula_pct = float("nan")
    if name == "stress":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            cfg_no_corr = create(
                about="Stress (no correlations)",
                unit="account",
                window=("2022-01", "2025-12", "monthly"),
                metrics=[{"name": f"m{i:02d}", "type": "score", "polarity": "positive"}
                         for i in range(20)],
                segments=[
                    {"name": "growth_seg",  "count": 500, "archetype": "growth"},
                    {"name": "decline_seg", "count": 500, "archetype": "decline"},
                ],
                connections=[],
                seed=SEED,
            )
        cfg_no_corr_serial = _switch_mode(cfg_no_corr, "serial")
        t_no_corr = _time_median(cfg_no_corr_serial)
        if t_serial > 0:
            copula_pct = (t_serial - t_no_corr) / t_serial * 100.0

    PERF_ROWS.append({
        "config":       name,
        "serial_s":     round(t_serial, 3),
        "vectorized_s": round(t_vec, 3) if not np.isnan(t_vec) else float("nan"),
        "speedup":      round(speedup, 2) if not np.isnan(speedup) else float("nan"),
        "copula_pct":   round(copula_pct, 1) if not np.isnan(copula_pct) else float("nan"),
    })

perf_df = pd.DataFrame(PERF_ROWS)
print(perf_df.to_string(index=False))


Config summary: 1,000 entities × 48 periods = 48,000 cells, 20 metrics, 3 tables. Estimated peak memory: ~116 MB.


config  serial_s  vectorized_s  speedup  copula_pct
  saas     9.196         2.410     3.82         NaN
stress   661.903         3.945   167.77        99.3


## Cell 9 â€” Fixture consistency (post-M127 only)

Skip in baseline run by setting `RUN_FIXTURE_CHECK = False` below.
After M127 regenerates `tests/fixtures/layer4_reference/`, verify that
the new fixtures are internally consistent: FK closure, PK uniqueness,
row-count matches, no unexpected NaNs in metric columns.

In [11]:
RUN_FIXTURE_CHECK = True  # flip to True after M127 regenerates fixtures

FIXTURE_ROWS = []
if RUN_FIXTURE_CHECK:
    fixtures_root = REPO_ROOT / "tests" / "fixtures" / "layer4_reference"
    for tdir in sorted(fixtures_root.iterdir()):
        if not tdir.is_dir():
            continue
        cfg_path = tdir / "config.yaml"
        if not cfg_path.exists():
            continue
        try:
            cfg = load_config(str(cfg_path))
        except Exception as e:
            FIXTURE_ROWS.append({
                "template": tdir.name, "check": "load_config",
                "passed": False, "detail": f"{type(e).__name__}: {e}"[:80],
            })
            continue
        csvs = sorted(tdir.glob("*.csv"))
        tables = {p.stem: pd.read_csv(p) for p in csvs}

        # PK uniqueness on every dim_*
        for tname, df in tables.items():
            if not tname.startswith("dim_"):
                continue
            pk_candidates = [c for c in df.columns if c.endswith("_id")]
            for pk in pk_candidates:
                ok = df[pk].is_unique
                FIXTURE_ROWS.append({
                    "template": tdir.name,
                    "check":    f"PK_unique[{tname}.{pk}]",
                    "passed":   bool(ok),
                    "detail":   "" if ok else f"{int(df[pk].duplicated().sum())} dupes",
                })

        # FK closure: every fct/evt FK column â†’ corresponding dim PK
        for tname, df in tables.items():
            if tname.startswith("dim_"):
                continue
            for col in df.columns:
                if not col.endswith("_id"):
                    continue
                stem = col[:-3]
                parent = f"dim_{stem}"
                if parent not in tables:
                    continue
                parent_pk = tables[parent].get(col)
                if parent_pk is None:
                    continue
                missing = (~df[col].isin(parent_pk)).sum()
                FIXTURE_ROWS.append({
                    "template": tdir.name,
                    "check":    f"FK[{tname}.{col}->{parent}]",
                    "passed":   missing == 0,
                    "detail":   "" if missing == 0 else f"{int(missing)} orphans",
                })

        # Metric-column null check on fct tables
        m_names = {m.name for m in cfg.metrics}
        for tname, df in tables.items():
            if not tname.startswith("fct_"):
                continue
            for col in df.columns:
                if col not in m_names:
                    continue
                nulls = int(df[col].isna().sum())
                FIXTURE_ROWS.append({
                    "template": tdir.name,
                    "check":    f"no_nulls[{tname}.{col}]",
                    "passed":   nulls == 0,
                    "detail":   "" if nulls == 0 else f"{nulls} nulls",
                })

if FIXTURE_ROWS:
    fixture_df = pd.DataFrame(FIXTURE_ROWS)
    print(fixture_df.to_string(index=False))
    fails = (~fixture_df["passed"]).sum()
    print(f"\nFailures: {int(fails)} / {len(fixture_df)}")
else:
    fixture_df = pd.DataFrame(columns=["template", "check", "passed", "detail"])
    print("Fixture check skipped (RUN_FIXTURE_CHECK=False).")


Config summary: 95 entities × 22 periods = 2,090 cells, 5 metrics, 6 tables. Estimated peak memory: ~100 MB.
Config summary: 85 entities × 36 periods = 3,060 cells, 5 metrics, 7 tables. Estimated peak memory: ~100 MB.
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:250: UserWarning: Correlation matrix was not positive definite. Adjusted 3 pairs to nearest valid values: absence_rate ↔ attrition_risk: 0.6100 → 0.5709 (Δ0.0391), engagement_index ↔ attrition_risk: -0.7200 → -0.6735 (Δ0.0465), performance_score ↔ attrition_risk: -0.5500 → -0.5149 (Δ0.0351)
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
Config summary: 100 entities × 24 periods = 2,400 cells, 9 metrics, 9 tables. Estimated peak memory: ~101 MB.
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:250: UserWarning: Correlation matrix was not positive definite. Adjusted 5 pairs to nearest valid values: bounce_rate

 template                                                     check  passed   detail
education                           PK_unique[dim_course.course_id]    True         
education                         PK_unique[dim_student.student_id]   False  4 dupes
education                         PK_unique[dim_student.dim_row_id]    True         
education               FK[bridge_enrollment.course_id->dim_course]    True         
education                   FK[evt_dropout.student_id->dim_student]    True         
education                FK[fct_engagement.student_id->dim_student]    True         
education                    FK[fct_grades.student_id->dim_student]    True         
education                      FK[fct_grades.course_id->dim_course]    True         
education                  no_nulls[fct_engagement.attendance_rate]    True         
education                      no_nulls[fct_engagement.study_hours]    True         
education                     no_nulls[fct_engagement.dropout_ris

Config summary: 90 entities × 24 periods = 2,160 cells, 6 metrics, 9 tables. Estimated peak memory: ~100 MB. Expected event rows (upper bound): ~10,800.
c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\main.py:250: UserWarning: Correlation matrix was not positive definite. Adjusted 3 pairs to nearest valid values: engagement ↔ churn_risk: -0.7500 → -0.6334 (Δ0.1166), engagement ↔ mrr: 0.8200 → 0.7332 (Δ0.0868), support_tickets ↔ churn_risk: 0.6800 → 0.6265 (Δ0.0535)
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## Cell 10 â€” Summary report

Concatenates all section tables into one markdown string. Prints to
output and writes `analysis/m127_verification_report.md` for diffing
against the prior run.

In [ ]:
def _to_md(df, max_rows=200):
    if df.empty:
        return "_(empty)_\n"
    if len(df) > max_rows:
        df = df.head(max_rows)
    return df.to_markdown(index=False) + "\n"


parts = []
parts.append("# M127 verification â€” measurement report\n")
parts.append(f"_Seed: {SEED}    Alt seed: {ALT_SEED}    Configs: "
             f"{', '.join(CONFIGS.keys())}_\n")

parts.append("## Statistical profile\n")
for tname, df in PROFILES.items():
    parts.append(f"### {tname}\n")
    parts.append(_to_md(df))

parts.append("## Shape recovery\n")
parts.append(_to_md(shape_df))

parts.append("## Correlation signs\n")
parts.append(_to_md(sign_df))

parts.append("## Degenerate centers\n")
parts.append(_to_md(DEGENERATE))

parts.append("## Determinism\n")
parts.append(_to_md(det_df))

parts.append("## FakerSource parity\n")
parts.append(_to_md(faker_df, max_rows=500))

parts.append("## Validation parity\n")
parts.append(_to_md(val_df))

parts.append("## Performance\n")
parts.append(_to_md(perf_df))

parts.append("## Fixture consistency\n")
parts.append(_to_md(fixture_df))

REPORT = "\n".join(parts)
print(REPORT)

out_path = REPO_ROOT / "analysis" / "m127_post.md"
out_path.write_text(REPORT, encoding="utf-8")
print(f"\nWrote {out_path}")
